# LAMBDA CSV-RDS

In [ ]:
import boto3
import pg8000
import csv
import time
from urllib.parse import unquote_plus

s3 = boto3.client('s3')

def to_int(value):
    try:
        return int(value) if value not in (None, "") else None
    except:
        return None

def to_float(value):
    try:
        return float(value) if value not in (None, "") else None
    except:
        return None

def to_bool(value):
    if value is None:
        return None
    return str(value).lower() == 'true'

def lambda_handler(event, context):
    start = time.time()
    conn = None
    cur = None

    try:
        host = "***************"
        dbname = "postgres"
        user = "postgres"
        password = "*********************"
        port = 5432

        bucket = event['Records'][0]['s3']['bucket']['name']
        key = unquote_plus(event['Records'][0]['s3']['object']['key'])

        print(bucket)
        print(key)

        obj = s3.get_object(Bucket=bucket, Key=key)
        data = obj['Body'].read().decode('utf-8-sig').splitlines()
        reader = csv.DictReader(data)

        print("Columnas CSV:", reader.fieldnames)

        rows = list(reader)

        if len(rows) == 0:
            return {
                "statusCode": 200,
                "body": "CSV vacío"
            }

        conn = pg8000.connect(
            host=host,
            database=dbname,
            user=user,
            password=password,
            port=port
        )

        cur = conn.cursor()

        values = []

        for r in rows:
            name = r.get('name')
            slug = r.get('slug')
            released = r.get('released')

            rating = to_float(r.get('rating'))
            metacritic = to_float(r.get('metacritic'))

            ratings_count = to_int(r.get('ratings_count'))
            reviews_text_count = to_int(r.get('reviews_text_count') or r.get('reviews_count'))

            playtime = to_int(r.get('playtime'))
            year = to_int(r.get('year'))

            added = to_int(r.get('added'))
            suggestions_count = to_int(r.get('suggestions_count'))

            num_platforms = to_int(r.get('num_platforms'))
            multiplayer = to_bool(r.get('multiplayer'))

            genres = r.get('genres')
            platforms = r.get('platforms')
            developers = r.get('developers')
            publishers = r.get('publishers')

            values.append((
                name, slug, released, rating, ratings_count,
                reviews_text_count, metacritic, added,
                suggestions_count, genres, platforms,
                developers, publishers, playtime, year,
                num_platforms, multiplayer
            ))

        cur.executemany(
            """
            INSERT INTO juegos (
                name, slug, released, rating, ratings_count,
                reviews_text_count, metacritic, added,
                suggestions_count, genres, platforms,
                developers, publishers, playtime, year,
                num_platforms, multiplayer
            ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            """,
            values
        )

        conn.commit()

        return {
            "statusCode": 200,
            "body": f"{len(values)} filas insertadas en {time.time() - start:.2f}s"
        }

    except Exception as e:
        if conn:
            conn.rollback()

        return {
            "statusCode": 500,
            "body": str(e)
        }

    finally:
        if cur:
            cur.close()
        if conn:
            conn.close()